In [ ]:
%pip install duckdb huggingface_hub risenlab-agentlogs

# Load the dataset

This notebook reads parquet from a local directory. Run **one** of the next two cells.

The sample under `data/dataset-sample/` is included in this repository and is enough to follow the examples.

In [ ]:
from pathlib import Path

from agentlogs.schema import assert_dataset_version

dataset_path = Path(".") / ".." / ".." / "data" / "dataset-sample"
assert_dataset_version(dataset_path)

For the full tables, download a Hugging Face snapshot into `data/dataset/` (files already present are skipped).

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download
from agentlogs.schema import assert_dataset_version

dataset_path = Path(".") / ".." / ".." / "data" / "dataset"
snapshot_download(
    repo_id="risenlab/agentlogs",
    repo_type="dataset",
    revision="v0.2",
    local_dir=dataset_path,
)
assert_dataset_version(dataset_path)

# Setup

In [ ]:
import duckdb

log_parts = sorted((dataset_path / "agent_session_logs").glob("*.parquet"))
table_path = {
    "repositories": dataset_path / "repositories" / "*.parquet",
    "agent_tasks": dataset_path / "agent_tasks" / "*.parquet",
    "agent_sessions": dataset_path / "agent_sessions" / "*.parquet",
    "agent_session_logs": dataset_path / "agent_session_logs" / "*.parquet",
    "agent_session_logs_shard": log_parts[0],
    "users": dataset_path / "users" / "*.parquet",
}
conn = duckdb.connect()

# Repositories

In [ ]:
n_repositories, n_repositories_with_tasks = conn.sql(
f"""
SELECT 
    COUNT(*), 
    COUNT(*) FILTER (WHERE length(agent_tasks) > 0),
FROM read_parquet('{table_path["repositories"]}')
"""
).fetchone()

print(f"# repositories: {n_repositories}")
print(f"# repositories with agent tasks: {n_repositories_with_tasks}")
print(f"% repositories with agent tasks: {100.0 * n_repositories_with_tasks / n_repositories:.2f}%")

# Agent tasks

In [ ]:
n_tasks, n_tasks_found, n_tasks_with_sessions = conn.sql(
f"""
SELECT
    COUNT(*),
    COUNT(*) FILTER (WHERE found),
    COUNT(*) FILTER (WHERE length(sessions) > 0),
FROM read_parquet('{table_path["agent_tasks"]}')
"""
).fetchone()

print(f"# tasks: {n_tasks}")
print(f"# tasks found: {n_tasks_found}")
print(f"% tasks found: {100.0 * n_tasks_found / n_tasks:.2f}%")
print(f"# tasks with agent sessions: {n_tasks_with_sessions}")
print(f"% tasks with agent sessions: {100.0 * n_tasks_with_sessions / n_tasks_found:.2f}%")

# Agent sessions

In [ ]:
n_sessions, n_logs_found = conn.sql(
f"""
SELECT 
    COUNT(*),
    COUNT(*) FILTER (WHERE log_found)
FROM read_parquet('{table_path["agent_sessions"]}')
"""
).fetchone()
n_sessions_with_nonempty_logs = conn.sql(
f"""
SELECT COUNT(DISTINCT session.id)
FROM read_parquet('{table_path["agent_session_logs"]}')
"""
).fetchone()[0]

print(f"# sessions: {n_sessions}")
print(f"# sessions with logs found: {n_logs_found}")
print(f"% sessions with logs found: {100.0 * n_logs_found / n_sessions:.2f}%")
print(f"# sessions with non-empty logs: {n_sessions_with_nonempty_logs}")
print(f"% sessions with non-empty logs: {100.0 * n_sessions_with_nonempty_logs / n_logs_found:.2f}%")

# Agent session logs

In [ ]:
n_log_entries, n_log_entries_parsed = conn.sql(
f"""
SELECT
    COUNT(*),
    COUNT(*) FILTER (WHERE parsed)
FROM read_parquet('{table_path["agent_session_logs"]}')
"""
).fetchone()

print(f"# log entries: {n_log_entries}")
print(f"# log entries parsed: {n_log_entries_parsed}")
print(f"% log entries parsed: {100.0 * n_log_entries_parsed / n_log_entries:.2f}%")

## Identify different kinds of tool calls

Uses the first `agent_session_logs` parquet shard only (see Setup).

In [ ]:
conn.sql(
"""
SET threads TO 4;
SET preserve_insertion_order = false;
"""
)

df = conn.sql(
f"""
WITH logs AS (
  SELECT data.choices AS choices
  FROM read_parquet('{table_path["agent_session_logs_shard"]}')
  WHERE parsed
    AND data IS NOT NULL
),

choices AS (
  SELECT choice
  FROM logs
  CROSS JOIN UNNEST(choices) AS c(choice)
),

delta_calls AS (
  SELECT
    'delta' AS source,
    COALESCE(tc.function_name, tc.custom_name) AS function_name,
    tc.type
  FROM choices
  CROSS JOIN UNNEST(choice.delta.tool_calls) AS t(tc)
  WHERE choice.delta.tool_calls IS NOT NULL
),

message_calls AS (
  SELECT
    'message' AS source,
    tc.function_name,
    tc.type
  FROM choices
  CROSS JOIN UNNEST(choice.message.tool_calls) AS t(tc)
  WHERE choice.message.tool_calls IS NOT NULL
),

all_calls AS (
  SELECT * FROM delta_calls
  UNION ALL
  SELECT * FROM message_calls
)

SELECT
  function_name,
  type,
  source,
  COUNT(*) AS n,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM all_calls
WHERE function_name IS NOT NULL
GROUP BY 1, 2, 3
ORDER BY n DESC;
"""
).fetchdf()
df

# Users

In [ ]:
n_users, n_users_found = conn.sql(
f"""
SELECT
    COUNT(*),
    COUNT(*) FILTER (WHERE found)
FROM read_parquet('{table_path["users"]}')
"""
).fetchone()

print(f"# users: {n_users}")
print(f"# users found: {n_users_found}")
print(f"% users found: {100.0 * n_users_found / n_users:.2f}%")